# 01 - Data Acquisition

Download satellite imagery and auxiliary data for Hyderabad urban heat analysis.

## Data Sources
- **Landsat 8/9**: Land Surface Temperature, NDVI, NDBI, MNDWI
- **ECOSTRESS**: High-resolution LST (70m)
- **MODIS**: Daily LST for temporal fusion
- **ERA5**: Meteorological forcing
- **SRTM DEM**: Elevation, slope
- **GHSL**: Built-up area, building height
- **SVF**: Sky View Factor from LiDAR/DEM

In [ ]:
import sys
sys.path.append('..')

import ee
import yaml
from pathlib import Path

from src.data.gee_interface import GEEInterface

In [ ]:
# Load configuration
with open('../config/config.yaml', 'r') as f:
    config = yaml.safe_load(f)

print(f"Study area: {config['study_area']['name']}")
print(f"Resolution: {config['resolution']}m")
print(f"Period: {config['temporal']['start_date']} to {config['temporal']['end_date']}")

In [ ]:
# Initialize GEE interface
gee = GEEInterface(config)

# Authenticate and initialize
# gee.authenticate()  # Run once
# gee.initialize()

In [ ]:
# Download Landsat data
landsat_data = gee.download_landsat(
    start_date='2024-04-01',
    end_date='2024-09-30',
    cloud_pct_max=20,
)
print(f"Downloaded {len(landsat_data)} Landsat scenes")

In [ ]:
# Download ERA5 data
era5_data = gee.download_era5(
    start_date='2024-04-01',
    end_date='2024-09-30',
    variables=['temperature_2m', 'relative_humidity_2m', 'wind_speed_10m',
              'surface_solar_radiation_downwards', 'surface_thermal_radiation_downwards'],
)
print(f"Downloaded ERA5 data: {era5_data}")

In [ ]:
# Download DEM data
dem_data = gee.download_dem()
print(f"Downloaded DEM data: {dem_data}")

In [ ]:
# Export to Google Drive
gee.export_to_drive(
    image=landsat_data,
    description='hyderabad_landsat_2024',
    folder='urban_heat_mitigation',
    scale=30,
    region=gee.hyderabad_bounds,
)
print("Export initiated. Check GEE Task Manager.")